# Multi-Modal PDF Parsing: Layout Analysis with Docling (GPU Accelerated)

In this notebook, we use **Docling** powered by our **NVIDIA GeForce RTX 4070 Ti SUPER (16 GB VRAM)** to analyse the layout of `input/jpmc_annualreport-2025.pdf`.

### Objectives:
1. Verify GPU acceleration with PyTorch CUDA.
2. Test Docling on sample pages representing different archetypes:
   - **Page 2**: Borderless Financial Highlights Table
   - **Page 8**: Pure Vector Financial Chart (20-year net income)
   - **Page 16**: Multi-column Shareholder Letter
   - **Page 76**: Three-Year Summary of Consolidated Financial Highlights
   - **Page 85**: Provision for Credit Losses Table (rule-based parser miss)
3. Save and load `doc` directly to/from JSON so we never have to re-run the converter.
4. Inspect the document layout tree (`DoclingDocument` structure):
   - Text blocks, column reading order, headings
   - Table structure detection (cells, rows, headers)
   - Figure/picture element detection on vector drawing pages
5. Export and evaluate formats (Markdown, JSON, Dict).

In [2]:
import sys
from pathlib import Path
import torch

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PDF_PATH = PROJECT_ROOT / "input" / "jpmc_annualreport-2025.pdf"
print(f"Target PDF: {PDF_PATH} (exists: {PDF_PATH.exists()})")

# Verify CUDA device
print(f"Python Executable: {sys.executable}")
print(f"CUDA Available:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:               {torch.cuda.get_device_name(0)}")
    print(f"VRAM:              {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")

Target PDF: C:\Users\user\Documents\Gen_AI\Multi-modal-RAG\input\jpmc_annualreport-2025.pdf (exists: True)
Python Executable: c:\Users\user\Documents\Gen_AI\Multi-modal-RAG\.venv\Scripts\python.exe
CUDA Available:    True
GPU:               NVIDIA GeForce RTX 4070 Ti SUPER
VRAM:              15.99 GB


In [3]:
import docling
from docling_core.types.doc import DoclingDocument
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    AcceleratorOptions,
    AcceleratorDevice
)

print(f"Docling version: {docling.__version__}")

c:\Users\user\Documents\Gen_AI\Multi-modal-RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Docling version: 2.130.0


### Step 1: Configure DocumentConverter with GPU Acceleration

We configure `PdfPipelineOptions` with `AcceleratorOptions(device=AcceleratorDevice.CUDA)` to offload layout models and TableFormer to the RTX 4070.

In [4]:
# Configure pipeline options with CUDA acceleration
pipeline_options = PdfPipelineOptions()
pipeline_options.do_table_structure = True
pipeline_options.do_ocr = False  # PDF has native digital text; disable OCR for max throughput
pipeline_options.generate_page_images = True  # Useful for visual inspection
pipeline_options.accelerator_options = AcceleratorOptions(
    num_threads=8,
    device=AcceleratorDevice.CUDA
)

# Benchmark pages: 2, 8, 16, 76, 85
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options,
            page_range=[2, 8, 16, 76, 85]
        )
    }
)

full_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

print("DocumentConverter initialized with CUDA acceleration for pages [2, 8, 16, 76, 85]!")

DocumentConverter initialized with CUDA acceleration for pages [2, 8, 16, 76, 85]!


### Step 2: Convert and Parse Benchmark Pages on GPU

In [6]:
import time

start_t = time.perf_counter()
print(f"Converting benchmark pages from {PDF_PATH.name} on GPU...")
conv_result = converter.convert(PDF_PATH)
elapsed = time.perf_counter() - start_t
doc = conv_result.document

print(f"Conversion complete in {elapsed:.2f} seconds!")

start_t = time.perf_counter()
print(f"Converting benchmark pages from {PDF_PATH.name} on GPU...")
full_conv_result = full_converter.convert(PDF_PATH)
elapsed = time.perf_counter() - start_t
full_doc = full_conv_result.document
print(f"Full conversion complete in {elapsed:.2f} seconds!")

Converting benchmark pages from jpmc_annualreport-2025.pdf on GPU...
2026-09-25 16:33:05,447 MatchingPostProcessor WARNING  3 of 124 pdf cells matched neither a row nor a column band of the 12x15 grid and were dropped from the table
2026-09-25 16:34:50,089 MatchingPostProcessor WARNING  1 of 184 pdf cells matched neither a row nor a column band of the 37x9 grid and were dropped from the table
Conversion complete in 136.73 seconds!
Converting benchmark pages from jpmc_annualreport-2025.pdf on GPU...


Loading weights: 100%|██████████| 770/770 [00:00<00:00, 10102.90it/s]


2026-09-25 16:35:22,700 MatchingPostProcessor WARNING  3 of 124 pdf cells matched neither a row nor a column band of the 12x15 grid and were dropped from the table
2026-09-25 16:37:09,891 MatchingPostProcessor WARNING  1 of 184 pdf cells matched neither a row nor a column band of the 37x9 grid and were dropped from the table
Full conversion complete in 139.85 seconds!


### Step 2b: Save and Reload `doc` to Avoid Re-running Converter

Docling documents can be serialized directly to a JSON file and reloaded instantly at any time.

In [9]:
# Output directory for parsed artifacts
OUTPUT_DIR = PROJECT_ROOT / "parsing" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOC_JSON_PATH = OUTPUT_DIR / "docling_parsed_benchmark.json"
FULL_DOC_JSON_PATH = OUTPUT_DIR / "full_docling_parsed.json"

# 1. Save doc to JSON
doc.save_as_json(DOC_JSON_PATH)

full_doc.save_as_json(FULL_DOC_JSON_PATH)
print(f"Saved parsed DoclingDocument to: {DOC_JSON_PATH} ({DOC_JSON_PATH.stat().st_size / 1024:.1f} KB)")

# 2. Reload doc from JSON (uncomment to reload without running Step 2 again!)
# doc = DoclingDocument.load_from_json(DOC_JSON_PATH)
# print("Successfully reloaded doc from JSON!")

Saved parsed DoclingDocument to: C:\Users\user\Documents\Gen_AI\Multi-modal-RAG\parsing\output\docling_parsed_benchmark.json (112480.0 KB)


### Step 3: Inspect Extracted Elements (Tables, Texts, Pictures)

Note:
- `table.export_to_dataframe(doc=doc)` avoids deprecation warnings.
- In Docling 2.x, `pic.caption_text(doc=doc)` returns the caption string.

In [11]:
print(f"Total texts extracted: {len(doc.texts)}")
print(f"Total tables detected: {len(doc.tables)}")
print(f"Total pictures/figures detected: {len(doc.pictures)}")

# Detailed inspection of detected tables
for i, table in enumerate(doc.tables):
    prov = table.prov[0] if table.prov else None
    page_no = prov.page_no if prov else "unknown"
    # Pass doc=doc to avoid deprecation warning
    df = table.export_to_dataframe(doc=doc)
    print(f"\n=== Table {i+1} on Page {page_no} (Shape: {df.shape[0]} rows x {df.shape[1]} cols) ===")
    print(df.head(6))

# Detailed inspection of pictures/figures (especially Page 8 vector chart)
for i, pic in enumerate(doc.pictures):
    prov = pic.prov[0] if pic.prov else None
    page_no = prov.page_no if prov else "unknown"
    caption = pic.caption_text(doc=doc)
    print(f"\n=== Picture/Figure {i+1} on Page {page_no} ===")
    print(f"Caption: '{caption}'" if caption else "Caption: (None)")
    print(f"Bounding box: {prov.bbox if prov else 'N/A'}")

Total texts extracted: 6355
Total tables detected: 295
Total pictures/figures detected: 73

=== Table 1 on Page 2 (Shape: 31 rows x 4 cols) ===
  As of or for the year ended December 31, (in millions, except per share, ratio data and employees)  \
0                     Selected income statement data                                                   
1                                  Total net revenue                                                   
2                          Total noninterest expense                                                   
3                           Pre-provision profit (a)                                                   
4                        Provision for credit losses                                                   
5                                         Net income                                                   

         2025           2024       2023  
0                                        
1   $ 182,447  $ 177,556 (g)  $ 158,104  
2

In [12]:
print(f"Total texts extracted: {len(full_doc.texts)}")
print(f"Total tables detected: {len(full_doc.tables)}")
print(f"Total pictures/figures detected: {len(full_doc.pictures)}")

Total texts extracted: 6355
Total tables detected: 295
Total pictures/figures detected: 73


### Step 4: Export to Structured Markdown & Validate Fidelity

In [7]:
md_content = doc.export_to_markdown()
md_path = OUTPUT_DIR / "benchmark_pages_docling.md"
md_path.write_text(md_content, encoding="utf-8")

print(f"Markdown output successfully saved to: {md_path}")
print(f"Total length: {len(md_content)} characters")
print("\n--- Sample Markdown Output (First 2000 chars) ---\n")
print(md_content[:2000])

Markdown output successfully saved to: C:\Users\user\Documents\Gen_AI\Multi-modal-RAG\parsing\output\benchmark_pages_docling.md
Total length: 2411953 characters

--- Sample Markdown Output (First 2000 chars) ---

<!-- image -->

## Life, Liberty and the Pursuit of Happiness

Annual Report 2025

## Financial Highlights

| As of or for the year ended December 31, (in millions, except per share, ratio data and employees)   | 2025        | 2024          | 2023        |
|------------------------------------------------------------------------------------------------------|-------------|---------------|-------------|
| Selected income statement data                                                                       |             |               |             |
| Total net revenue                                                                                    | $ 182,447   | $ 177,556 (g) | $ 158,104   |
| Total noninterest expense                                                        